In [1]:
import duckdb
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r'C:\Users\Tobi\Desktop\CSHW\TransitKPIFramework')
DB_PATH      = PROJECT_ROOT / 'data' / 'transit_kpi.duckdb'

con = duckdb.connect(str(DB_PATH))

print(f'Database: {DB_PATH}')
print()
print('Tables:')
print(con.sql('SHOW TABLES').df().to_string(index=False))

Database: C:\Users\Tobi\Desktop\CSHW\TransitKPIFramework\data\transit_kpi.duckdb

Tables:
              name
          calendar
    calendar_dates
            gtfsrt
otp_specifications
        stop_times
       trip_delays
             trips


In [2]:
required_tables = ['gtfsrt', 'trips', 'calendar', 'calendar_dates', 'stop_times']

existing = set(con.sql('SHOW TABLES').df()['name'].tolist())
missing = [t for t in required_tables if t not in existing]
if missing:
    raise RuntimeError(
        f'Missing required tables: {missing}. '
        f'Run notebook 02 first to populate them.'
    )

print('All required tables present. Row counts:')
for tbl in required_tables:
    n = con.sql(f'SELECT COUNT(*) AS n FROM {tbl}').fetchone()[0]
    print(f'  {tbl:18s} {n:>14,} rows')

print()
print('gtfsrt service date range:')
print(con.sql("""
    SELECT
        MIN(service_date) AS first_date,
        MAX(service_date) AS last_date,
        COUNT(DISTINCT service_date) AS n_dates
    FROM gtfsrt
""").df().to_string(index=False))

All required tables present. Row counts:
  gtfsrt                402,682,884 rows
  trips                      35,316 rows
  calendar                       28 rows
  calendar_dates                688 rows
  stop_times              2,045,404 rows

gtfsrt service date range:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

first_date  last_date  n_dates
2026-04-04 2026-04-25       22


In [3]:
# For each service_date in gtfsrt, resolve which service_ids are active.
# Day-of-week mapping: ISODOW returns 1=Monday through 7=Sunday.

print('Active service_ids per service_date:')
print(con.sql("""
    WITH dates AS (
        SELECT DISTINCT service_date FROM gtfsrt
    ),
    regular AS (
        SELECT d.service_date, c.service_id
        FROM dates d
        JOIN calendar c
          ON CAST(REPLACE(d.service_date, '-', '') AS INTEGER)
             BETWEEN c.start_date AND c.end_date
         AND CASE ISODOW(CAST(d.service_date AS DATE))
                 WHEN 1 THEN c.monday
                 WHEN 2 THEN c.tuesday
                 WHEN 3 THEN c.wednesday
                 WHEN 4 THEN c.thursday
                 WHEN 5 THEN c.friday
                 WHEN 6 THEN c.saturday
                 WHEN 7 THEN c.sunday
             END = 1
    ),
    additions AS (
        SELECT d.service_date, cd.service_id
        FROM dates d
        JOIN calendar_dates cd
          ON cd.date = CAST(REPLACE(d.service_date, '-', '') AS INTEGER)
         AND cd.exception_type = 1
    ),
    removals AS (
        SELECT d.service_date, cd.service_id
        FROM dates d
        JOIN calendar_dates cd
          ON cd.date = CAST(REPLACE(d.service_date, '-', '') AS INTEGER)
         AND cd.exception_type = 2
    ),
    combined AS (
        SELECT service_date, service_id FROM regular
        UNION
        SELECT service_date, service_id FROM additions
        EXCEPT
        SELECT service_date, service_id FROM removals
    )
    SELECT
        service_date,
        ISODOW(CAST(service_date AS DATE)) AS dow,
        COUNT(DISTINCT service_id) AS active_services
    FROM combined
    GROUP BY service_date
    ORDER BY service_date
""").df().to_string(index=False))

Active service_ids per service_date:
service_date  dow  active_services
  2026-04-04    6               11
  2026-04-05    7               13
  2026-04-06    1               19
  2026-04-07    2               19
  2026-04-08    3               19
  2026-04-09    4               19
  2026-04-10    5               19
  2026-04-11    6               11
  2026-04-12    7               13
  2026-04-13    1               19
  2026-04-14    2               19
  2026-04-15    3               19
  2026-04-16    4               19
  2026-04-17    5               19
  2026-04-18    6               11
  2026-04-19    7               13
  2026-04-20    1               19
  2026-04-21    2               19
  2026-04-22    3               19
  2026-04-23    4               19
  2026-04-24    5               19
  2026-04-25    6               11


In [4]:
# Diagnostic: classify trips on routes 23 and 47 by their FIRST scheduled stop.
# The Owl filter we apply is: first_stop_seconds < 86400 --- trips beginning
# before midnight are 'day service' and kept (even if their later stops cross
# midnight as 24:xx times in static GTFS).

print('First-stop-time distribution for routes 23 and 47 trips:')
print(con.sql("""
    WITH trip_first_stop AS (
        SELECT
            t.trip_id,
            t.route_id,
            t.service_id,
            MIN(st.arrival_seconds) AS first_stop_seconds
        FROM trips t
        JOIN stop_times st USING (trip_id)
        WHERE t.route_id IN ('23', '47')
        GROUP BY t.trip_id, t.route_id, t.service_id
    )
    SELECT
        route_id,
        CASE
            WHEN first_stop_seconds < 14400  THEN '00_pre_4am'
            WHEN first_stop_seconds < 21600  THEN '01_early_AM_4_6'
            WHEN first_stop_seconds < 75600  THEN '02_high_freq_6_21'
            WHEN first_stop_seconds < 84600  THEN '03_late_eve_21_2330'
            WHEN first_stop_seconds < 86400  THEN '04_boundary_2330_24'
            WHEN first_stop_seconds < 100800 THEN '05_owl_24_28'
            ELSE                                  '06_owl_28_plus'
        END AS bucket,
        COUNT(*) AS n_trips,
        MIN(first_stop_seconds) AS min_secs,
        MAX(first_stop_seconds) AS max_secs
    FROM trip_first_stop
    GROUP BY route_id, bucket
    ORDER BY route_id, bucket
""").df().to_string(index=False))

First-stop-time distribution for routes 23 and 47 trips:
route_id              bucket  n_trips  min_secs  max_secs
      23          00_pre_4am        3     12000     12000
      23     01_early_AM_4_6       43     14400     21480
      23   02_high_freq_6_21      393     21960     75480
      23 03_late_eve_21_2330       39     75720     84420
      23 04_boundary_2330_24        6     85260     85860
      23        05_owl_24_28       39     87180     99120
      47          00_pre_4am       27      1560     13560
      47     01_early_AM_4_6       25     15120     21420
      47   02_high_freq_6_21      405     21720     75480
      47 03_late_eve_21_2330       30     75780     83640
      47 04_boundary_2330_24        6     84660     85380
      47        05_owl_24_28        6     86940     89040


In [5]:
# Build matched_stops: every gtfsrt record joined to its scheduled stop time,
# with all filters applied EXCEPT dedup. This intermediate table is shared
# across all three snapshot-rule trip_delays_* tables.
#
# Filters applied here:
#   - Routes 23 and 47
#   - schedule_relationship = 0 (SCHEDULED)
#   - Day-service trips (first scheduled stop < 24:00:00)
#   - Per-service_date active service resolution (with calendar_dates removals)
#
# Each row is one (trip_id, stop_sequence, service_date, snapshot_ts) tuple ---
# i.e. one prediction. A given stop event has many rows, one per snapshot it
# appeared in. The downstream dedup rules choose which row to keep.

con.execute('DROP TABLE IF EXISTS matched_stops')

con.execute("""
    CREATE TABLE matched_stops AS
    WITH dates AS (
        SELECT DISTINCT service_date FROM gtfsrt
    ),
    active_services AS (
        SELECT d.service_date, c.service_id
        FROM dates d
        JOIN calendar c
          ON CAST(REPLACE(d.service_date, '-', '') AS INTEGER)
             BETWEEN c.start_date AND c.end_date
         AND CASE ISODOW(CAST(d.service_date AS DATE))
                 WHEN 1 THEN c.monday
                 WHEN 2 THEN c.tuesday
                 WHEN 3 THEN c.wednesday
                 WHEN 4 THEN c.thursday
                 WHEN 5 THEN c.friday
                 WHEN 6 THEN c.saturday
                 WHEN 7 THEN c.sunday
             END = 1

        UNION

        SELECT d.service_date, cd.service_id
        FROM dates d
        JOIN calendar_dates cd
          ON cd.date = CAST(REPLACE(d.service_date, '-', '') AS INTEGER)
         AND cd.exception_type = 1
    ),
    trip_first_stop AS (
        SELECT trip_id, MIN(arrival_seconds) AS first_stop_seconds
        FROM stop_times
        GROUP BY trip_id
    ),
    raw_matched AS (
        SELECT
            rt.trip_id,
            rt.route_id,
            rt.service_date,
            rt.snapshot_ts,
            rt.stop_sequence,
            rt.stop_id,
            rt.schedule_relationship,
            rt.predicted_unix,
            rt.predicted_ts,
            rt.midnight_unix,
            t.direction_id,
            st.arrival_time           AS scheduled_arrival_time,
            st.arrival_seconds        AS scheduled_arrival_seconds,
            st.departure_time         AS scheduled_departure_time,
            st.departure_seconds      AS scheduled_departure_seconds,
            st.timepoint
        FROM gtfsrt rt
        JOIN trips t
            ON rt.trip_id = t.trip_id
        JOIN active_services a
            ON t.service_id   = a.service_id
           AND a.service_date = rt.service_date
        JOIN stop_times st
            ON rt.trip_id        = st.trip_id
           AND rt.stop_sequence  = st.stop_sequence
        JOIN trip_first_stop tfs
            ON tfs.trip_id = rt.trip_id
        WHERE rt.route_id IN ('23', '47')
          AND rt.schedule_relationship = 0
          AND tfs.first_stop_seconds < 86400
          AND NOT EXISTS (
              SELECT 1 FROM calendar_dates cd
              WHERE cd.service_id     = t.service_id
                AND cd.date           = CAST(REPLACE(rt.service_date, '-', '') AS INTEGER)
                AND cd.exception_type = 2
          )
    )
    -- Compute scheduled_unix with day-boundary nearest-day correction.
    -- This is shared across all dedup rules.
    SELECT
        *,
        CASE
            WHEN ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds - 86400))
               < ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds))
             AND ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds - 86400))
               < ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds + 86400))
            THEN midnight_unix + scheduled_arrival_seconds - 86400
            WHEN ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds + 86400))
               < ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds))
            THEN midnight_unix + scheduled_arrival_seconds + 86400
            ELSE midnight_unix + scheduled_arrival_seconds
        END AS scheduled_unix,
        -- Convert snapshot_ts (ISO string in UTC) to Unix for ordering rules
        EPOCH(CAST(snapshot_ts AS TIMESTAMP)) AS snapshot_unix
    FROM raw_matched
""")

print('matched_stops built. Row count:')
print(con.sql('SELECT COUNT(*) AS rows FROM matched_stops').df().to_string(index=False))
print()
print('Distinct stop events (trip_id, stop_sequence, service_date):')
print(con.sql("""
    SELECT COUNT(*) AS distinct_stop_events
    FROM (
        SELECT trip_id, stop_sequence, service_date
        FROM matched_stops
        GROUP BY trip_id, stop_sequence, service_date
    )
""").df().to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

matched_stops built. Row count:
    rows
24145195

Distinct stop events (trip_id, stop_sequence, service_date):
 distinct_stop_events
               745669


In [6]:
# Snapshot rule: 'closest'
# For each (trip_id, stop_sequence, service_date), keep the snapshot whose
# predicted_unix is closest to scheduled_unix.
#
# Interpretation: 'the prediction most likely to be accurate.' Tends to bias
# OTP upward because near-arrival predictions converge to actual arrival, and
# any prediction within the on-time window gets selected even if it was
# issued an hour earlier. This is the original methodology from the v1 paper
# pipeline and is included here for comparison.

con.execute('DROP TABLE IF EXISTS trip_delays_closest')

con.execute("""
    CREATE TABLE trip_delays_closest AS
    WITH ranked AS (
        SELECT
            *,
            ROUND((predicted_unix - scheduled_unix) / 60.0, 1) AS delay_minutes,
            ROW_NUMBER() OVER (
                PARTITION BY trip_id, stop_sequence, service_date
                ORDER BY ABS(predicted_unix - scheduled_unix)
            ) AS snapshot_rank
        FROM matched_stops
    )
    SELECT
        trip_id, route_id, direction_id, service_date, snapshot_ts,
        stop_sequence, stop_id, schedule_relationship, timepoint,
        scheduled_arrival_time, scheduled_arrival_seconds,
        scheduled_departure_time, scheduled_departure_seconds,
        scheduled_unix, predicted_unix, predicted_ts,
        delay_minutes
    FROM ranked
    WHERE snapshot_rank = 1
""")

print('trip_delays_closest:')
print(con.sql('SELECT COUNT(*) AS rows FROM trip_delays_closest').df().to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

trip_delays_closest:
  rows
745669


In [7]:
# Snapshot rule: 'last_before'
# For each (trip_id, stop_sequence, service_date), keep the latest snapshot
# whose snapshot_ts is BEFORE scheduled_unix.
#
# Interpretation: 'what a rider would have seen on the SEPTA app right before
# the bus was scheduled to arrive.' This is the rider-experience prediction.
#
# Caveat: Some stop events may have ZERO qualifying snapshots --- e.g. if all
# our predictions for that stop arrived after the scheduled time. Those events
# do not appear in this table. The trip_delays_last_before sample is therefore
# typically smaller than trip_delays_closest. This sample-size difference is
# itself a methodological observation worth reporting.

con.execute('DROP TABLE IF EXISTS trip_delays_last_before')

con.execute("""
    CREATE TABLE trip_delays_last_before AS
    WITH eligible AS (
        SELECT *
        FROM matched_stops
        WHERE snapshot_unix < scheduled_unix
    ),
    ranked AS (
        SELECT
            *,
            ROUND((predicted_unix - scheduled_unix) / 60.0, 1) AS delay_minutes,
            ROW_NUMBER() OVER (
                PARTITION BY trip_id, stop_sequence, service_date
                ORDER BY snapshot_unix DESC
            ) AS snapshot_rank
        FROM eligible
    )
    SELECT
        trip_id, route_id, direction_id, service_date, snapshot_ts,
        stop_sequence, stop_id, schedule_relationship, timepoint,
        scheduled_arrival_time, scheduled_arrival_seconds,
        scheduled_departure_time, scheduled_departure_seconds,
        scheduled_unix, predicted_unix, predicted_ts,
        delay_minutes
    FROM ranked
    WHERE snapshot_rank = 1
""")

print('trip_delays_last_before:')
print(con.sql('SELECT COUNT(*) AS rows FROM trip_delays_last_before').df().to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

trip_delays_last_before:
  rows
729768


In [8]:
# Snapshot rule: 'latest'
# For each (trip_id, stop_sequence, service_date), keep the chronologically
# last snapshot --- i.e. the one with the maximum snapshot_ts.
#
# Interpretation: 'the final prediction GTFS-RT made for this stop event.'
# As a bus approaches and passes a stop, predictions converge to actual
# arrival, so the last snapshot is the closest GTFS-RT-only proxy for ground
# truth. This is the recommended default for outside analysts who want the
# best estimate of actual performance.

con.execute('DROP TABLE IF EXISTS trip_delays_latest')

con.execute("""
    CREATE TABLE trip_delays_latest AS
    WITH ranked AS (
        SELECT
            *,
            ROUND((predicted_unix - scheduled_unix) / 60.0, 1) AS delay_minutes,
            ROW_NUMBER() OVER (
                PARTITION BY trip_id, stop_sequence, service_date
                ORDER BY snapshot_unix DESC
            ) AS snapshot_rank
        FROM matched_stops
    )
    SELECT
        trip_id, route_id, direction_id, service_date, snapshot_ts,
        stop_sequence, stop_id, schedule_relationship, timepoint,
        scheduled_arrival_time, scheduled_arrival_seconds,
        scheduled_departure_time, scheduled_departure_seconds,
        scheduled_unix, predicted_unix, predicted_ts,
        delay_minutes
    FROM ranked
    WHERE snapshot_rank = 1
""")

print('trip_delays_latest:')
print(con.sql('SELECT COUNT(*) AS rows FROM trip_delays_latest').df().to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

trip_delays_latest:
  rows
745669


In [9]:
# Side-by-side comparison of the three snapshot rules.
# Same routes, same dates, same trips --- only the dedup rule differs.
#
# Expected pattern:
#   - 'closest' has the most rows (no filter beyond dedup)
#   - 'latest' has the same row count as 'closest' (both rank all snapshots)
#   - 'last_before' has fewer rows (drops stop events with no qualifying snapshot)
#   - OTP under 'closest' is generally highest (selection bias toward on-time)
#   - OTP under 'last_before' is generally lowest (rider-experience uncertainty)
#   - OTP under 'latest' falls between

print('Row counts by snapshot rule:')
print(con.sql("""
    SELECT 'closest'      AS rule, COUNT(*) AS rows FROM trip_delays_closest
    UNION ALL
    SELECT 'last_before',          COUNT(*)         FROM trip_delays_last_before
    UNION ALL
    SELECT 'latest',               COUNT(*)         FROM trip_delays_latest
""").df().to_string(index=False))

print()
print('OTP by snapshot rule and route (-1 to +5 min window):')
print(con.sql("""
    WITH all_rules AS (
        SELECT 'closest'     AS rule, route_id, delay_minutes FROM trip_delays_closest
        UNION ALL
        SELECT 'last_before',         route_id, delay_minutes FROM trip_delays_last_before
        UNION ALL
        SELECT 'latest',              route_id, delay_minutes FROM trip_delays_latest
    )
    SELECT
        rule,
        route_id,
        COUNT(*) AS stop_events,
        ROUND(
            SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5 THEN 1 ELSE 0 END)
            * 100.0 / COUNT(*), 1
        ) AS otp_pct,
        ROUND(AVG(delay_minutes), 2) AS mean_delay,
        ROUND(MEDIAN(delay_minutes), 2) AS median_delay
    FROM all_rules
    GROUP BY rule, route_id
    ORDER BY route_id, rule
""").df().to_string(index=False))

Row counts by snapshot rule:
       rule   rows
    closest 745669
last_before 729768
     latest 745669

OTP by snapshot rule and route (-1 to +5 min window):
       rule route_id  stop_events  otp_pct  mean_delay  median_delay
    closest       23       377990     86.7        1.02           0.0
last_before       23       369962     60.0        3.15           1.8
     latest       23       377990     55.4        4.24           2.3
    closest       47       367679     85.8        1.18           0.1
last_before       47       359806     55.3        4.28           2.8
     latest       47       367679     48.5        5.85           3.7


In [10]:
# Delay distribution buckets, broken out by snapshot rule.
# Surfaces how the dedup choice shifts the distribution shape.

print('Delay distribution by snapshot rule:')
print(con.sql("""
    WITH all_rules AS (
        SELECT 'closest'     AS rule, delay_minutes FROM trip_delays_closest
        UNION ALL
        SELECT 'last_before',         delay_minutes FROM trip_delays_last_before
        UNION ALL
        SELECT 'latest',              delay_minutes FROM trip_delays_latest
    )
    SELECT
        rule,
        CASE
            WHEN delay_minutes < -60 THEN 'a_bad_match_neg'
            WHEN delay_minutes < -30 THEN 'b_very_early'
            WHEN delay_minutes < -1  THEN 'c_early'
            WHEN delay_minutes <= 5  THEN 'd_on_time'
            WHEN delay_minutes <= 15 THEN 'e_late'
            WHEN delay_minutes <= 60 THEN 'f_very_late'
            ELSE                          'g_bad_match_pos'
        END AS bucket,
        COUNT(*) AS records
    FROM all_rules
    GROUP BY rule, bucket
    ORDER BY rule, bucket
""").df().to_string(index=False))

Delay distribution by snapshot rule:
       rule          bucket  records
    closest         c_early    48890
    closest       d_on_time   643051
    closest          e_late    42605
    closest     f_very_late    11123
last_before         c_early    95507
last_before       d_on_time   420997
last_before          e_late   180829
last_before     f_very_late    32435
     latest         c_early    92942
     latest       d_on_time   387637
     latest          e_late   204355
     latest     f_very_late    59131
     latest g_bad_match_pos     1604


In [11]:
# Coverage diagnostic: how many stop events does last_before drop?
# A stop event is dropped if all its snapshots have snapshot_ts >= scheduled_unix.
# This happens for late-evening predictions where SEPTA only publishes a
# few snapshots, all close to actual arrival (which is after scheduled).

print('Coverage of last_before relative to closest:')
print(con.sql("""
    WITH closest_events AS (
        SELECT trip_id, stop_sequence, service_date
        FROM trip_delays_closest
    ),
    last_before_events AS (
        SELECT trip_id, stop_sequence, service_date
        FROM trip_delays_last_before
    ),
    counts AS (
        SELECT
            (SELECT COUNT(*) FROM closest_events)     AS closest_n,
            (SELECT COUNT(*) FROM last_before_events) AS last_before_n
    )
    SELECT
        closest_n,
        last_before_n,
        closest_n - last_before_n AS dropped,
        ROUND(last_before_n * 100.0 / closest_n, 2) AS coverage_pct
    FROM counts
""").df().to_string(index=False))

Coverage of last_before relative to closest:
 closest_n  last_before_n  dropped  coverage_pct
    745669         729768    15901         97.87


In [ ]:
con.close()
print('Connection closed')